# LAD Figure Workflow

This notebook rebuilds LAD figures from cached analysis outputs under `results/04_lad_analysis/tables`.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from utils.figures_utils import (
    LAD_FIGURE_OUTPUT_DIR,
    LAD_METRIC_LABELS,
    add_lad_tool_labels,
    apply_lad_tool_order,
    get_lad_association_metrics_df,
    get_lad_combined_summary_df,
    get_lad_null_summary_df,
    get_lad_profile_outputs_df,
    get_lad_reference_summary_df,
    get_lad_sample_genomes_df,
    get_lad_unique_lad_exports_df,
    get_lad_unique_lad_summary_df,
    lad_label_palette,
    save_publication_figure,
    style_publication_axis,
)

sns.set_theme(style="whitegrid")
OUTPUT_DIR = Path(LAD_FIGURE_OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SAVE_PDF = False


In [2]:
metrics_df = add_lad_tool_labels(apply_lad_tool_order(get_lad_association_metrics_df().copy()))
combined_summary_df = add_lad_tool_labels(apply_lad_tool_order(get_lad_combined_summary_df().copy()))
null_summary_df = add_lad_tool_labels(apply_lad_tool_order(get_lad_null_summary_df().copy()))
unique_lad_exports_df = get_lad_unique_lad_exports_df().copy()
unique_lad_summary_df = get_lad_unique_lad_summary_df().copy()
profile_outputs_df = get_lad_profile_outputs_df().copy()
sample_genomes_df = get_lad_sample_genomes_df().copy()
reference_summary_df = get_lad_reference_summary_df().copy()

required_frames = {
    "metrics_df": metrics_df,
    "combined_summary_df": combined_summary_df,
    "null_summary_df": null_summary_df,
    "unique_lad_exports_df": unique_lad_exports_df,
}
missing = [name for name, frame in required_frames.items() if frame.empty]
if missing:
    raise RuntimeError(
        "Missing or empty LAD cached outputs: " + ", ".join(missing) + ". "
        "Run analysis/04_lad_analysis/01_run_lad.py first."
    )

print(f"Loaded {len(metrics_df):,} LAD metric rows")
print(f"Loaded {len(combined_summary_df):,} combined LAD summary rows")
print(f"Loaded {len(null_summary_df):,} LAD null-summary rows")
print(f"Loaded {len(unique_lad_exports_df):,} unique-LAD export rows")
display(sample_genomes_df)
display(reference_summary_df)


Loaded 264 LAD metric rows
Loaded 88 combined LAD summary rows
Loaded 264 LAD null-summary rows
Loaded 6 unique-LAD export rows


,sample,sample_id,genome
0,ESO26.wgbs,ESO26,hg38
1,TE5.wgbs,TE5,hg38
2,WGBS_colon-primary-tumor_1_meth,WGBS_colon-primary-tumor_1_meth,hg38


,genome,chrom_sizes_path,lad_clean_path,occupancy_bw_path,signal_bw_path,n_lad_regions,lad_total_bp,n_lad_regions_raw,n_lad_regions_filtered,lad_total_bp_raw,lad_total_bp_filtered,min_lad_beta
0,hg38,/scratch/general/vast/u0914269/methylseg_resul...,/scratch/general/vast/u0914269/methylseg_resul...,/scratch/general/vast/u0914269/methylseg_resul...,/scratch/general/vast/u0914269/methylseg_resul...,6741,1138991407,6741,6741,1138991407,1138991407,0.0


In [3]:
palette = lad_label_palette()

def metric_label(metric_name: str) -> str:
    return LAD_METRIC_LABELS.get(metric_name, metric_name.replace("_", " "))

def plot_lad_metric_bar(plot_df: pd.DataFrame, value_col: str, title: str, ylabel: str):
    ordered_df = plot_df.sort_values(value_col, ascending=False).reset_index(drop=True)
    ordered_labels = ordered_df["tool_label"].astype(str).tolist()
    fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
    sns.barplot(
        data=ordered_df,
        x="tool_label",
        y=value_col,
        order=ordered_labels,
        palette=[palette[label] for label in ordered_labels],
        ax=ax,
    )
    style_publication_axis(ax, title=title, ylabel=ylabel, x_label_rotation=40)
    ax.grid(axis="y", alpha=0.25)
    return fig, ax


In [4]:
combined_output_dir = OUTPUT_DIR / "combined_metrics"
for metric_name, metric_df in combined_summary_df.groupby("metric", sort=False):
    fig, _ = plot_lad_metric_bar(
        metric_df,
        value_col="mean_score",
        title=f"Combined LAD: {metric_label(metric_name)}",
        ylabel="Mean score",
    )
    save_publication_figure(
        fig,
        combined_output_dir,
        f"combined.{metric_name}.png",
        save_pdf=SAVE_PDF,
    )
    plt.close(fig)

print(f"Saved combined LAD figures to {combined_output_dir}")


/tmp/ipykernel_1944371/2551639603.py:10: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/tmp/ipykernel_1944371/2551639603.py:10: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/tmp/ipykernel_1944371/2551639603.py:10: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/tmp/ipykernel_1944371/2551639603.py:10: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/tmp/ipykernel_1944371/2551639603.py:10: Fut

Saved combined LAD figures to /uufs/chpc.utah.edu/common/home/u0914269/clement/projects/20260624_methylseg/results/figures/05_lad/combined_metrics


In [5]:
per_sample_output_dir = OUTPUT_DIR / "per_sample_metrics"
for (sample, metric_name), metric_df in metrics_df.groupby(["sample", "metric"], sort=False):
    fig, _ = plot_lad_metric_bar(
        metric_df,
        value_col="score",
        title=f"{sample}: {metric_label(metric_name)}",
        ylabel="Score",
    )
    save_publication_figure(
        fig,
        per_sample_output_dir,
        f"{sample}.{metric_name}.png",
        save_pdf=SAVE_PDF,
    )
    plt.close(fig)

print(f"Saved per-sample LAD figures to {per_sample_output_dir}")


/tmp/ipykernel_1944371/2551639603.py:10: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/tmp/ipykernel_1944371/2551639603.py:10: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/tmp/ipykernel_1944371/2551639603.py:10: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/tmp/ipykernel_1944371/2551639603.py:10: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/tmp/ipykernel_1944371/2551639603.py:10: Fut

Saved per-sample LAD figures to /uufs/chpc.utah.edu/common/home/u0914269/clement/projects/20260624_methylseg/results/figures/05_lad/per_sample_metrics


In [6]:
null_output_dir = OUTPUT_DIR / "null_model"
null_agg_df = (
    null_summary_df.groupby(["tool", "tool_label", "metric"], as_index=False)
    .agg(
        enrichment_vs_null=("enrichment_vs_null", "mean"),
        z_score_vs_null=("z_score_vs_null", "mean"),
        n_samples=("sample", "nunique"),
    )
)

for metric_name, metric_df in null_agg_df.groupby("metric", sort=False):
    fig, _ = plot_lad_metric_bar(
        metric_df,
        value_col="enrichment_vs_null",
        title=f"Null enrichment: {metric_label(metric_name)}",
        ylabel="Observed / null mean",
    )
    save_publication_figure(
        fig,
        null_output_dir,
        f"null_enrichment.{metric_name}.png",
        save_pdf=SAVE_PDF,
    )
    plt.close(fig)

    fig, _ = plot_lad_metric_bar(
        metric_df,
        value_col="z_score_vs_null",
        title=f"Null z-score: {metric_label(metric_name)}",
        ylabel="Z-score vs null",
    )
    save_publication_figure(
        fig,
        null_output_dir,
        f"null_zscore.{metric_name}.png",
        save_pdf=SAVE_PDF,
    )
    plt.close(fig)

print(f"Saved LAD null-model figures to {null_output_dir}")


/tmp/ipykernel_1944371/2551639603.py:10: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/tmp/ipykernel_1944371/2551639603.py:10: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/tmp/ipykernel_1944371/2551639603.py:10: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/tmp/ipykernel_1944371/2551639603.py:10: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/tmp/ipykernel_1944371/2551639603.py:10: Fut

Saved LAD null-model figures to /uufs/chpc.utah.edu/common/home/u0914269/clement/projects/20260624_methylseg/results/figures/05_lad/null_model


In [7]:
display(unique_lad_summary_df)
display(unique_lad_exports_df)
display(profile_outputs_df)


,comparison_tool,reference_methylseg_tool,sample,genome,n_unique_lads,n_unique_regions
0,dnmtools,methylseg,ESO26.wgbs,hg38,1157,63
1,dnmtools,methylseg,TE5.wgbs,hg38,1004,55
2,dnmtools,methylseg,WGBS_colon-primary-tumor_1_meth,hg38,30,5
3,dnmtools_array,methylseg_hm450k,ESO26.wgbs,hg38,229,33
4,dnmtools_array,methylseg_hm450k,TE5.wgbs,hg38,680,55
5,dnmtools_array,methylseg_hm450k,WGBS_colon-primary-tumor_1_meth,hg38,1,1


,comparison_tool,reference_methylseg_tool,sample,genome,n_unique_lads,export_path
0,dnmtools,methylseg,ESO26.wgbs,hg38,1157,/scratch/general/vast/u0914269/methylseg_resul...
1,dnmtools,methylseg,TE5.wgbs,hg38,1004,/scratch/general/vast/u0914269/methylseg_resul...
2,dnmtools,methylseg,WGBS_colon-primary-tumor_1_meth,hg38,30,/scratch/general/vast/u0914269/methylseg_resul...
3,dnmtools_array,methylseg_hm450k,ESO26.wgbs,hg38,229,/scratch/general/vast/u0914269/methylseg_resul...
4,dnmtools_array,methylseg_hm450k,TE5.wgbs,hg38,680,/scratch/general/vast/u0914269/methylseg_resul...
5,dnmtools_array,methylseg_hm450k,WGBS_colon-primary-tumor_1_meth,hg38,1,/scratch/general/vast/u0914269/methylseg_resul...


,sample,sample_id,genome,occupancy_matrix_path,occupancy_sorted_regions_path,occupancy_profile_path,occupancy_heatmap_path,signal_matrix_path,signal_sorted_regions_path,signal_profile_path,signal_heatmap_path,include_heatmaps,visualized_tools
